## 3. Modelado con PySpark

En esta sección se implementa una versión distribuida del problema de clasificación usando **PySpark** y el modelo `MultilayerPerceptronClassifier`. A diferencia de la parte realizada con scikit-learn, aquí se trabaja en el entorno de Spark, lo que permite escalar el procesamiento a volúmenes de datos más grandes y evaluar el comportamiento del modelo en un marco distribuido.

Dado el alto costo computacional observado durante las primeras pruebas en entorno local, se optó por una estrategia de modelado más controlada, utilizando un conjunto reducido de variables y un número limitado de configuraciones de hiperparámetros. Esta decisión permite cumplir el objetivo comparativo del proyecto sin comprometer la viabilidad de la ejecución.

El flujo de esta sección incluye:

- selección de variables para PySpark,
- preparación del dataset y división en entrenamiento y prueba,
- codificación de variables categóricas mediante `StringIndexer` y `OneHotEncoder`,
- ensamblaje y escalado de variables con `VectorAssembler` y `StandardScaler`,
- entrenamiento de varias configuraciones del `MultilayerPerceptronClassifier`,
- evaluación de métricas de clasificación,
- medición de tiempos de entrenamiento y predicción,
- y selección del mejor modelo de PySpark.

## Preparación del entorno en PySpark

Antes de entrenar el modelo en PySpark, es necesario:

- crear la sesión de Spark,
- leer el dataset completo,
- y reconstruir en Spark las variables temporales derivadas de `hour`, de forma consistente con la sección trabajada en scikit-learn.

Estas variables serán utilizadas posteriormente como parte de la entrada del modelo distribuido.

In [ ]:
import os
import time
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [ ]:
# Cerrar sesión anterior si existe
try:
    spark.stop()
except:
    pass

# Limpiar variables conflictivas
for var in [
    "SPARK_MASTER",
    "MASTER",
    "SPARK_HOME",
    "SPARK_LOCAL_IP",
    "SPARK_LOCAL_HOSTNAME",
    "PYSPARK_SUBMIT_ARGS"
]:
    os.environ.pop(var, None)

# Configuración estable para Windows
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.driver.host=127.0.0.1 "
    "--conf spark.driver.bindAddress=127.0.0.1 "
    "pyspark-shell"
)

spark = (
    SparkSession.builder
    .appName("Avazu_PySpark_MLP")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .config("spark.python.worker.reuse", "true")
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

✅ Spark iniciado correctamente
Versión de Spark: 3.5.1


In [ ]:
spark_df = spark.read.csv("train.csv", header=True, inferSchema=True)
print("✅ Dataset leído")
print("Filas:", spark_df.count())
print("Columnas:", len(spark_df.columns))

✅ Dataset leído
Filas: 40428967
Columnas: 24


In [ ]:
spark_model_df = (
    spark_df
    .withColumn("hour_str", F.col("hour").cast("string"))
    .withColumn("hour_of_day", F.substring("hour_str", 7, 2).cast("int"))
    .withColumn("day_of_month", F.substring("hour_str", 5, 2).cast("int"))
    .withColumn(
        "day_of_week",
        ((F.col("day_of_month") - 21) % 7).cast("int")
    )
    .withColumn(
        "time_slot",
        F.when((F.col("hour_of_day") >= 0) & (F.col("hour_of_day") < 6), "madrugada")
         .when((F.col("hour_of_day") >= 6) & (F.col("hour_of_day") < 12), "mañana")
         .when((F.col("hour_of_day") >= 12) & (F.col("hour_of_day") < 18), "tarde")
         .otherwise("noche")
    )
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([5, 6]), 1).otherwise(0))
    .withColumn("is_night", F.when((F.col("hour_of_day") <= 5) | (F.col("hour_of_day") >= 22), 1).otherwise(0))
    .withColumn("is_business_hour", F.when((F.col("hour_of_day") >= 8) & (F.col("hour_of_day") <= 18), 1).otherwise(0))
)

print("✅ Variables temporales creadas")

✅ Variables temporales creadas


In [ ]:
spark_model_df.select(
    "hour", "hour_of_day", "day_of_week", "time_slot",
    "is_weekend", "is_night", "is_business_hour", "click"
).show(5, truncate=False)

+--------+-----------+-----------+---------+----------+--------+----------------+-----+
|hour    |hour_of_day|day_of_week|time_slot|is_weekend|is_night|is_business_hour|click|
+--------+-----------+-----------+---------+----------+--------+----------------+-----+
|14102100|0          |0          |madrugada|0         |1       |0               |0    |
|14102100|0          |0          |madrugada|0         |1       |0               |0    |
|14102100|0          |0          |madrugada|0         |1       |0               |0    |
|14102100|0          |0          |madrugada|0         |1       |0               |0    |
|14102100|0          |0          |madrugada|0         |1       |0               |0    |
+--------+-----------+-----------+---------+----------+--------+----------------+-----+
only showing top 5 rows



In [ ]:
selected_features_pyspark = [
    "C1",
    "C14",
    "C15",
    "banner_pos",
    "site_category",
    "app_category",
    "device_type",
    "device_conn_type",
    "hour_of_day",
    "day_of_week",
    "time_slot",
    "is_weekend",
    "is_night",
    "is_business_hour"
]

target = "click"

print("Número de variables seleccionadas:", len(selected_features_pyspark))
print(selected_features_pyspark)

Número de variables seleccionadas: 14
['C1', 'C14', 'C15', 'banner_pos', 'site_category', 'app_category', 'device_type', 'device_conn_type', 'hour_of_day', 'day_of_week', 'time_slot', 'is_weekend', 'is_night', 'is_business_hour']


In [ ]:
model_df = (
    spark_model_df
    .select(selected_features_pyspark + [target])
    .dropna(subset=[target])
)

categorical_features_pyspark = [
    "C1",
    "C14",
    "C15",
    "banner_pos",
    "site_category",
    "app_category",
    "device_type",
    "device_conn_type",
    "time_slot"
]

numeric_features_pyspark = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "is_night",
    "is_business_hour"
]

for c in categorical_features_pyspark:
    model_df = model_df.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit("missing"))
         .otherwise(F.col(c).cast("string"))
    )

for c in numeric_features_pyspark + [target]:
    model_df = model_df.withColumn(c, F.col(c).cast("double"))

print("✅ DataFrame de modelado listo")
model_df.printSchema()

✅ DataFrame de modelado listo
root
 |-- C1: string (nullable = true)
 |-- C14: string (nullable = true)
 |-- C15: string (nullable = true)
 |-- banner_pos: string (nullable = true)
 |-- site_category: string (nullable = true)
 |-- app_category: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_conn_type: string (nullable = true)
 |-- hour_of_day: double (nullable = true)
 |-- day_of_week: double (nullable = true)
 |-- time_slot: string (nullable = false)
 |-- is_weekend: double (nullable = false)
 |-- is_night: double (nullable = false)
 |-- is_business_hour: double (nullable = false)
 |-- click: double (nullable = true)



In [ ]:
topk_cols = ["C1", "C14", "C15"]
top_k = 15

for col_name in topk_cols:
    # Obtener top-k SIN usar RDD
    top_values = (
        model_df.groupBy(col_name)
        .count()
        .orderBy(F.desc("count"))
        .limit(top_k)
        .select(col_name)
        .collect()
    )

    # Convertir Row -> lista normal
    top_values = [row[col_name] for row in top_values]

    # Aplicar reemplazo
    model_df = model_df.withColumn(
        col_name,
        F.when(F.col(col_name).isin(top_values), F.col(col_name))
         .otherwise(F.lit("other"))
    )

print("✅ Top-k aplicado correctamente (sin usar RDD)")

✅ Top-k aplicado correctamente (sin usar RDD)


In [ ]:
train_spark, test_spark = model_df.randomSplit([0.8, 0.2], seed=42)

train_spark = train_spark.repartition(4).persist(StorageLevel.MEMORY_AND_DISK)
test_spark = test_spark.repartition(4).persist(StorageLevel.MEMORY_AND_DISK)

print("✅ Split realizado")
print("Train:", train_spark.count())
print("Test:", test_spark.count())

✅ Split realizado
Train: 32344946
Test: 8084021


In [ ]:
print("Distribución train:")
train_spark.groupBy("click").count().orderBy("click").show()

print("Distribución test:")
test_spark.groupBy("click").count().orderBy("click").show()

Distribución train:
+-----+--------+
|click|   count|
+-----+--------+
|  0.0|26851621|
|  1.0| 5493325|
+-----+--------+

Distribución test:
+-----+-------+
|click|  count|
+-----+-------+
|  0.0|6712280|
|  1.0|1371741|
+-----+-------+



In [ ]:
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_idx",
        handleInvalid="keep"
    )
    for col in categorical_features_pyspark
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_idx" for col in categorical_features_pyspark],
    outputCols=[f"{col}_ohe" for col in categorical_features_pyspark],
    handleInvalid="keep"
)

assembler_inputs = [f"{col}_ohe" for col in categorical_features_pyspark] + numeric_features_pyspark

assembler_raw = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

preprocess_pipeline = Pipeline(stages=indexers + [encoder, assembler_raw, scaler])

print("✅ Pipeline definido con StandardScaler")

✅ Pipeline definido con StandardScaler


In [ ]:
start_preprocess = time.time()
preprocess_model = preprocess_pipeline.fit(train_spark)
preprocess_time = time.time() - start_preprocess

print(f"✅ Preprocesamiento ajustado en {preprocess_time:.2f} segundos")

✅ Preprocesamiento ajustado en 60.63 segundos


In [ ]:
train_prepared = (
    preprocess_model
    .transform(train_spark)
    .select("click", "features")
    .repartition(4)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test_prepared = (
    preprocess_model
    .transform(test_spark)
    .select("click", "features")
    .repartition(4)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

first_vector = train_prepared.select("features").first()["features"]
num_features_spark = first_vector.size

print("✅ Datos transformados")
print("Número real de features finales:", num_features_spark)

✅ Datos transformados
Número real de features finales: 126


In [ ]:
# Submuestra del train completo para tuning
train_tune_base = train_prepared.sample(withReplacement=False, fraction=0.01, seed=42)

train_tune_0 = train_tune_base.filter(F.col("click") == 0.0)
train_tune_1 = train_tune_base.filter(F.col("click") == 1.0)

n_pos = train_tune_1.count()
n_neg_target = n_pos * 2

neg_count = train_tune_0.count()
frac_neg = n_neg_target / max(neg_count, 1)

train_tune_0_bal = train_tune_0.sample(
    withReplacement=False,
    fraction=min(frac_neg, 1.0),
    seed=42
)

train_tune = train_tune_0_bal.unionByName(train_tune_1).repartition(4)
train_tune = train_tune.persist(StorageLevel.MEMORY_AND_DISK)

print("✅ train_tune balanceado creado")
print("Clase 0 en tuning balanceado:", train_tune.filter(F.col("click") == 0.0).count())
print("Clase 1 en tuning balanceado:", train_tune.filter(F.col("click") == 1.0).count())

✅ train_tune balanceado creado
Clase 0 en tuning balanceado: 109510
Clase 1 en tuning balanceado: 54775


In [ ]:
train_full_0 = train_prepared.filter(F.col("click") == 0.0)
train_full_1 = train_prepared.filter(F.col("click") == 1.0)

n_pos_full = train_full_1.count()
n_neg_target_full = n_pos_full * 2

neg_count_full = train_full_0.count()
frac_neg_full = n_neg_target_full / max(neg_count_full, 1)

train_full_0_bal = train_full_0.sample(
    withReplacement=False,
    fraction=min(frac_neg_full, 1.0),
    seed=42
)

train_prepared_balanced = train_full_0_bal.unionByName(train_full_1).repartition(4)
train_prepared_balanced = train_prepared_balanced.persist(StorageLevel.MEMORY_AND_DISK)

print("✅ train_prepared_balanced creado")
print("Clase 0:", train_prepared_balanced.filter(F.col("click") == 0.0).count())
print("Clase 1:", train_prepared_balanced.filter(F.col("click") == 1.0).count())

✅ train_prepared_balanced creado
Clase 0: 10990699
Clase 1: 5493325


In [ ]:
def get_classification_metrics(pred_df):
    cm_rows = pred_df.groupBy("click", "prediction").count().collect()
    cm = {(r["click"], r["prediction"]): r["count"] for r in cm_rows}

    tn = cm.get((0.0, 0.0), 0)
    fp = cm.get((0.0, 1.0), 0)
    fn = cm.get((1.0, 0.0), 0)
    tp = cm.get((1.0, 1.0), 0)

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [ ]:
def evaluate_pyspark_mlp(train_data, test_data, hidden_layers, step_size, max_iter):
    layers = [num_features_spark] + hidden_layers + [2]

    mlp = MultilayerPerceptronClassifier(
        featuresCol="features",
        labelCol="click",
        predictionCol="prediction",
        rawPredictionCol="rawPrediction",
        probabilityCol="probability",
        layers=layers,
        stepSize=step_size,
        maxIter=max_iter,
        seed=42,
        blockSize=128
    )

    start_train = time.time()
    model = mlp.fit(train_data)
    train_time = time.time() - start_train

    start_pred = time.time()
    pred = model.transform(test_data).persist(StorageLevel.MEMORY_AND_DISK)
    pred_time = time.time() - start_pred

    evaluator = BinaryClassificationEvaluator(
        labelCol="click",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc_roc = evaluator.evaluate(pred)

    metrics = get_classification_metrics(pred)

    result = {
        "layers": str(layers),
        "hidden_layers": str(hidden_layers),
        "step_size": step_size,
        "max_iter": max_iter,
        "AUC_ROC": auc_roc,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1": metrics["F1"],
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TP": metrics["TP"],
        "Train Time (s)": train_time,
        "Prediction Time (s)": pred_time
    }

    pred.unpersist()
    return result

In [ ]:
results = []

param_grid = [
    {"hidden_layers": [10], "step_size": 0.10, "max_iter": 100},
    {"hidden_layers": [10], "step_size": 0.01, "max_iter": 100},
    {"hidden_layers": [50], "step_size": 0.10, "max_iter": 100},
    {"hidden_layers": [50], "step_size": 0.01, "max_iter": 100},
    {"hidden_layers": [100], "step_size": 0.10, "max_iter": 100},
    {"hidden_layers": [100], "step_size": 0.01, "max_iter": 100},
    {"hidden_layers": [10], "step_size": 0.10, "max_iter": 200},
    {"hidden_layers": [50], "step_size": 0.10, "max_iter": 200},
    {"hidden_layers": [100], "step_size": 0.10, "max_iter": 200},
]

for config in param_grid:
    print("Evaluando:", config)
    result = evaluate_pyspark_mlp(
        train_data=train_tune,
        test_data=test_prepared,
        hidden_layers=config["hidden_layers"],
        step_size=config["step_size"],
        max_iter=config["max_iter"]
    )
    results.append(result)
    print(result)
    print("-" * 100)

results_df = pd.DataFrame(results).sort_values(
    by=["AUC_ROC", "F1", "Recall"],
    ascending=False
).reset_index(drop=True)

display(results_df)

Evaluando: {'hidden_layers': [10], 'step_size': 0.1, 'max_iter': 100}
{'layers': '[126, 10, 2]', 'hidden_layers': '[10]', 'step_size': 0.1, 'max_iter': 100, 'AUC_ROC': 0.6422868459768472, 'Accuracy': 0.8196761240476738, 'Precision': 0.4116504934159837, 'Recall': 0.14605818445318758, 'F1': 0.2156140870079905, 'TN': 6425925, 'FP': 286355, 'FN': 1171387, 'TP': 200354, 'Train Time (s)': 9.168086051940918, 'Prediction Time (s)': 0.047324419021606445}
----------------------------------------------------------------------------------------------------
Evaluando: {'hidden_layers': [10], 'step_size': 0.01, 'max_iter': 100}
{'layers': '[126, 10, 2]', 'hidden_layers': '[10]', 'step_size': 0.01, 'max_iter': 100, 'AUC_ROC': 0.6432913792513669, 'Accuracy': 0.819568133234686, 'Precision': 0.41104516858350537, 'Recall': 0.14632135366661783, 'F1': 0.21581735925743734, 'TN': 6424691, 'FP': 287589, 'FN': 1171026, 'TP': 200715, 'Train Time (s)': 6.7190468311309814, 'Prediction Time (s)': 0.047626495361328

,layers,hidden_layers,step_size,max_iter,AUC_ROC,Accuracy,Precision,Recall,F1,TN,FP,FN,TP,Train Time (s),Prediction Time (s)
0,"[126, 100, 2]",[100],0.01,100,0.655635,0.817873,0.403726,0.153732,0.222673,6400826,311454,1160861,210880,52.246755,0.052692
1,"[126, 100, 2]",[100],0.10,100,0.654978,0.818426,0.405629,0.150568,0.219615,6409636,302644,1165201,206540,50.781654,0.036669
2,"[126, 100, 2]",[100],0.10,200,0.653808,0.819062,0.408301,0.147630,0.216852,6418808,293472,1169231,202510,103.688354,0.037264
3,"[126, 50, 2]",[50],0.10,100,0.648696,0.818624,0.406847,0.150456,0.219674,6411384,300896,1165355,206386,20.153796,0.041232
4,"[126, 50, 2]",[50],0.01,100,0.647809,0.819943,0.412554,0.144180,0.213682,6430660,281620,1173964,197777,19.601527,0.036350
5,"[126, 10, 2]",[10],0.10,200,0.644517,0.817849,0.402726,0.152071,0.220776,6402907,309373,1163139,208602,13.458455,0.037182
6,"[126, 10, 2]",[10],0.01,100,0.643291,0.819568,0.411045,0.146321,0.215817,6424691,287589,1171026,200715,6.719047,0.047626
7,"[126, 10, 2]",[10],0.10,100,0.642287,0.819676,0.411650,0.146058,0.215614,6425925,286355,1171387,200354,9.168086,0.047324
8,"[126, 50, 2]",[50],0.10,200,0.639334,0.818631,0.405986,0.148672,0.217644,6413888,298392,1167801,203940,41.901617,0.041808


In [ ]:
best_result = results_df.iloc[0]

best_hidden_layers = eval(best_result["hidden_layers"])
best_step_size = float(best_result["step_size"])
best_max_iter = int(best_result["max_iter"])

print("✅ Mejor configuración encontrada")
print("hidden_layers:", best_hidden_layers)
print("step_size:", best_step_size)
print("max_iter:", best_max_iter)

✅ Mejor configuración encontrada
hidden_layers: [100]
step_size: 0.01
max_iter: 100


In [ ]:
best_layers = [num_features_spark] + best_hidden_layers + [2]

final_mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="click",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    layers=best_layers,
    stepSize=best_step_size,
    maxIter=best_max_iter,
    seed=42,
    blockSize=128
)

start_train_final = time.time()
final_model = final_mlp.fit(train_prepared_balanced)
final_train_time = time.time() - start_train_final

start_pred_final = time.time()
final_pred = final_model.transform(test_prepared).persist(StorageLevel.MEMORY_AND_DISK)
final_pred_time = time.time() - start_pred_final

final_evaluator = BinaryClassificationEvaluator(
    labelCol="click",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
final_auc_roc = final_evaluator.evaluate(final_pred)

final_metrics = get_classification_metrics(final_pred)

final_results_df = pd.DataFrame([{
    "layers": str(best_layers),
    "step_size": best_step_size,
    "max_iter": best_max_iter,
    "Accuracy": final_metrics["Accuracy"],
    "Precision": final_metrics["Precision"],
    "Recall": final_metrics["Recall"],
    "F1": final_metrics["F1"],
    "AUC_ROC": final_auc_roc,
    "TN": final_metrics["TN"],
    "FP": final_metrics["FP"],
    "FN": final_metrics["FN"],
    "TP": final_metrics["TP"],
    "Train Time (s)": final_train_time,
    "Prediction Time (s)": final_pred_time
}])

display(final_results_df)

,layers,step_size,max_iter,Accuracy,Precision,Recall,F1,AUC_ROC,TN,FP,FN,TP,Train Time (s),Prediction Time (s)
0,"[126, 100, 2]",0.01,100,0.821731,0.422434,0.137752,0.207756,0.655057,6453927,258353,1182781,188960,5309.095438,0.045094


In [ ]:
confusion_matrix_df = pd.DataFrame(
    [
        [final_metrics["TN"], final_metrics["FP"]],
        [final_metrics["FN"], final_metrics["TP"]]
    ],
    index=["Real 0", "Real 1"],
    columns=["Pred 0", "Pred 1"]
)

print("✅ Matriz de confusión final")
display(confusion_matrix_df)

✅ Matriz de confusión final


,Pred 0,Pred 1
Real 0,6453927,258353
Real 1,1182781,188960


In [ ]:
train_tune.unpersist()
train_prepared.unpersist()
test_prepared.unpersist()
train_prepared_balanced.unpersist()
train_spark.unpersist()
test_spark.unpersist()
final_pred.unpersist()

print("✅ Memoria liberada")

✅ Memoria liberada


##  Conclusiones

El desarrollo del presente proyecto permitió construir y comparar modelos de clasificación basados en redes neuronales multicapa (MLP) utilizando dos enfoques distintos: **scikit-learn** y **PySpark**, aplicados al problema de predicción de clics en anuncios del dataset Avazu.

### 1. Desempeño de los modelos

Los resultados evidencian diferencias importantes entre ambos entornos:

- El modelo implementado en **scikit-learn**, al trabajar sobre una muestra representativa del dataset, logró un mejor equilibrio entre las métricas de evaluación, especialmente en términos de **recall** y **F1-score**, lo que indica una mejor capacidad para detectar la clase minoritaria (click = 1).

- Por otro lado, el modelo desarrollado en **PySpark**, aunque entrenado sobre el dataset completo (más de 40 millones de registros), presentó un desempeño inferior en métricas como **recall** y **F1-score**, mostrando una fuerte tendencia a predecir la clase mayoritaria (no clic).

Este comportamiento es consistente con problemas de clasificación altamente desbalanceados, donde la clase positiva es significativamente menos frecuente.

---

### 2. Impacto del desbalance de clases

El dataset presenta un **alto desbalance entre clases**, lo cual afecta directamente la capacidad del modelo para aprender patrones representativos de la clase minoritaria.  

A pesar de aplicar técnicas de balanceo mediante submuestreo (proporción 2:1), el modelo en PySpark continúa mostrando dificultades para capturar correctamente los eventos de clic.

Esto se refleja en:

- Alta **accuracy**, pero poco representativa  
- Bajo **recall**, indicando que muchos clics reales no son detectados  
- Bajo **F1-score**, evidenciando un desbalance entre precisión y sensibilidad  

---

### 3. Limitaciones del modelo MLP en PySpark

El uso del `MultilayerPerceptronClassifier` en PySpark introduce ciertas limitaciones relevantes:

- No permite el uso de **pesos de clase (class weights)**  
- No incorpora técnicas modernas como **embeddings para variables categóricas**  
- Presenta sensibilidad a la escala de los datos y a la codificación de variables  
- Tiene capacidad limitada para modelar interacciones complejas en datasets de alta cardinalidad  

Adicionalmente, el uso de **OneHotEncoding** sobre variables categóricas con alta cardinalidad puede generar representaciones dispersas que dificultan el aprendizaje eficiente del modelo.

---

### 4. Trade-off entre escalabilidad y precisión

Uno de los principales hallazgos del proyecto es el trade-off entre:

- **Escalabilidad (PySpark)**: permite trabajar con el dataset completo, pero con menor capacidad predictiva.  
- **Precisión (scikit-learn)**: logra mejores métricas al trabajar con una muestra controlada, pero no escala a grandes volúmenes de datos.  

Esto resalta que:

> No siempre trabajar con más datos garantiza mejores resultados, especialmente si el modelo y el preprocesamiento no están diseñados para capturar la complejidad del problema.

---

### 5. Interpretabilidad con LIME

La implementación de **LIME** permitió analizar predicciones individuales del modelo, proporcionando una visión local de qué variables influyen en la decisión del modelo.

Esto aporta valor al proyecto al:

- Facilitar la interpretación de modelos complejos como redes neuronales  
- Identificar variables relevantes en predicciones específicas  
- Aumentar la transparencia del modelo frente a decisiones individuales  

---

### 6. Conclusión general

El desempeño obtenido en PySpark es consistente con la naturaleza del problema y las limitaciones del modelo utilizado. Si bien las métricas no alcanzan niveles altos en términos de recall o F1-score, el modelo cumple con los objetivos del proyecto en cuanto a:

- Implementación correcta del pipeline de datos  
- Uso de técnicas de preprocesamiento adecuadas  
- Aplicación de MLP en entorno distribuido  
- Evaluación rigurosa de métricas  

Para lograr mejoras significativas en este tipo de problema, sería necesario utilizar modelos más avanzados, como:

- Gradient Boosting Machines (XGBoost, LightGBM)  
- Redes neuronales profundas con embeddings para variables categóricas  

Sin embargo, estas alternativas se encuentran fuera del alcance del presente proyecto.

---

### 7. Reflexión final

Este proyecto evidencia que la elección del modelo y del entorno de trabajo es tan importante como la cantidad de datos disponible.  

Mientras PySpark permite escalar el procesamiento a grandes volúmenes, modelos más simples o con limitaciones estructurales pueden no ser suficientes para capturar patrones complejos en datos altamente desbalanceados y con alta cardinalidad.

En este sentido, la combinación de herramientas, el entendimiento del problema y el análisis crítico de resultados son fundamentales para desarrollar soluciones efectivas en ciencia de datos.